# Exercise 2: Strategy Pattern for Missing Values Handling

**Objective:** Learn the Strategy Pattern - one interface, many interchangeable behaviors.

**Concept:** Different strategies to handle missing data (drop, fill mean, fill mode, etc.) that can be flexibly switched without changing the main code.

## Step 1: Import Libraries

In [13]:
import pandas as pd
import numpy as np
from abc import ABC, abstractmethod
from typing import Protocol

## Step 2: Load the Data

In [14]:
# Load the dataset
df = pd.read_csv('sample_data (2).csv')

print("Original Dataset:")
print(df)
print("\nMissing Values Count:")
print(df.isnull().sum())
print("\nData Types:")
print(df.dtypes)

Original Dataset:
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0
5   6    Frank  30.0      175.0       85.0  Los Angeles    NaN
6   7    Grace   NaN      162.0       60.0      Chicago   92.0

Missing Values Count:
id           0
name         0
age          3
height_cm    1
weight_kg    1
city         0
score        2
dtype: int64

Data Types:
id             int64
name          object
age          float64
height_cm    float64
weight_kg    float64
city          object
score        float64
dtype: object


## Step 3: Define the Strategy Pattern

### Abstract Base Class: MissingValueStrategy

In [15]:
class MissingValueStrategy(ABC):
    """Abstract base class for missing value handling strategies."""
    
    @abstractmethod
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        """Handle missing values in the dataframe.
        
        Args:
            df: Input dataframe with missing values
            
        Returns:
            DataFrame with missing values handled
        """
        pass
    
    def __str__(self):
        return self.__class__.__name__

### Strategy 1: DropMissing

In [16]:
class DropMissing(MissingValueStrategy):
    """Strategy to drop rows with any missing values."""
    
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        """Drop all rows containing missing values."""
        result = df.dropna()
        print(f"Dropped {len(df) - len(result)} rows with missing values.")
        return result

### Strategy 2: FillMean

In [17]:
class FillMean(MissingValueStrategy):
    """Strategy to fill missing values with the mean of numeric columns."""
    
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        """Fill missing values with column means for numeric columns."""
        result = df.copy()
        
        # Get numeric columns only
        numeric_cols = result.select_dtypes(include=[np.number]).columns
        
        for col in numeric_cols:
            if result[col].isnull().any():
                mean_value = result[col].mean()
                result[col].fillna(mean_value, inplace=True)
                print(f"Filled {col} with mean: {mean_value:.2f}")
        
        # For non-numeric columns, we could drop rows or use a different strategy
        # Here we'll just leave them as is
        return result

### Strategy 3: FillMode

In [18]:
class FillMode(MissingValueStrategy):
    """Strategy to fill missing values with the mode (most frequent value)."""
    
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        """Fill missing values with column mode."""
        result = df.copy()
        
        for col in result.columns:
            if result[col].isnull().any():
                mode_value = result[col].mode()
                if len(mode_value) > 0:
                    result[col].fillna(mode_value[0], inplace=True)
                    print(f"Filled {col} with mode: {mode_value[0]}")
        
        return result

## Step 4: DataCleaner Class

This class uses composition to apply the chosen strategy.

In [19]:
class DataCleaner:
    """Context class that applies a missing value handling strategy."""
    
    def __init__(self, strategy: MissingValueStrategy):
        """Initialize with a specific strategy.
        
        Args:
            strategy: A MissingValueStrategy instance
        """
        self._strategy = strategy
    
    def set_strategy(self, strategy: MissingValueStrategy):
        """Change the strategy at runtime.
        
        Args:
            strategy: A new MissingValueStrategy instance
        """
        self._strategy = strategy
        print(f"Strategy changed to: {strategy}")
    
    def clean_data(self, df: pd.DataFrame) -> pd.DataFrame:
        """Clean the dataframe using the current strategy.
        
        Args:
            df: Input dataframe
            
        Returns:
            Cleaned dataframe
        """
        print(f"\nApplying strategy: {self._strategy}")
        print("-" * 50)
        return self._strategy.handle(df)

## Step 5: Test Different Strategies

### Test 1: Drop Missing Strategy

In [20]:
# Create a cleaner with DropMissing strategy
cleaner = DataCleaner(DropMissing())
df_dropped = cleaner.clean_data(df)

print("\nResult after dropping missing values:")
print(df_dropped)
print(f"\nShape: {df_dropped.shape}")


Applying strategy: DropMissing
--------------------------------------------------
Dropped 5 rows with missing values.

Result after dropping missing values:
   id   name   age  height_cm  weight_kg      city  score
0   1  Alice  29.0      165.0       68.0  New York   85.0
4   5    Eva  27.0      160.0       55.0  New York   88.0

Shape: (2, 7)


### Test 2: Fill Mean Strategy

In [21]:
# Change strategy to FillMean
cleaner.set_strategy(FillMean())
df_mean_filled = cleaner.clean_data(df)

print("\nResult after filling with mean:")
print(df_mean_filled)
print("\nMissing values after filling with mean:")
print(df_mean_filled.isnull().sum())

Strategy changed to: FillMean

Applying strategy: FillMean
--------------------------------------------------
Filled age with mean: 30.25
Filled height_cm with mean: 167.00
Filled weight_kg with mean: 70.00
Filled score with mean: 86.00

Result after filling with mean:
   id     name    age  height_cm  weight_kg         city  score
0   1    Alice  29.00      165.0       68.0     New York   85.0
1   2      Bob  30.25      172.0       70.0  Los Angeles   90.0
2   3  Charlie  35.00      168.0       72.0      Chicago   86.0
3   4    David  30.25      167.0       80.0      Houston   75.0
4   5      Eva  27.00      160.0       55.0     New York   88.0
5   6    Frank  30.00      175.0       85.0  Los Angeles   86.0
6   7    Grace  30.25      162.0       60.0      Chicago   92.0

Missing values after filling with mean:
id           0
name         0
age          0
height_cm    0
weight_kg    0
city         0
score        0
dtype: int64


C:\Users\User\AppData\Local\Temp\ipykernel_12360\940559035.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  result[col].fillna(mean_value, inplace=True)


### Test 3: Fill Mode Strategy

In [22]:
# Change strategy to FillMode
cleaner.set_strategy(FillMode())
df_mode_filled = cleaner.clean_data(df)

print("\nResult after filling with mode:")
print(df_mode_filled)
print("\nMissing values after filling with mode:")
print(df_mode_filled.isnull().sum())

Strategy changed to: FillMode

Applying strategy: FillMode
--------------------------------------------------
Filled age with mode: 27.0
Filled height_cm with mode: 160.0
Filled weight_kg with mode: 55.0
Filled score with mode: 75.0

Result after filling with mode:
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob  27.0      172.0       55.0  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago   75.0
3   4    David  27.0      160.0       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0
5   6    Frank  30.0      175.0       85.0  Los Angeles   75.0
6   7    Grace  27.0      162.0       60.0      Chicago   92.0

Missing values after filling with mode:
id           0
name         0
age          0
height_cm    0
weight_kg    0
city         0
score        0
dtype: int64


C:\Users\User\AppData\Local\Temp\ipykernel_12360\4106858363.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  result[col].fillna(mode_value[0], inplace=True)


## Summary

**Benefits of the Strategy Pattern:**

1. **Flexibility:** Easy to switch between different algorithms at runtime
2. **Extensibility:** New strategies can be added without modifying existing code
3. **Encapsulation:** Each strategy is encapsulated in its own class
4. **Open/Closed Principle:** Open for extension, closed for modification
5. **Single Responsibility:** Each strategy class has one reason to change

**When to use:**
- Multiple algorithms for the same task
- Need to switch algorithms at runtime
- Want to avoid conditional statements for algorithm selection